In [1]:
import os
os.environ["OPENCV_LOG_LEVEL"] = "FATAL" # 只输出致命错误，屏蔽 WARNING
import cv2
import threading
from socket import *
import time
from flask import Flask, Response
import logging
from LOBOROBOT import LOBOROBOT  # 载入统一的机器人库

In [2]:
# ================= 1. 初始化硬件 =================
print("初始化底盘与摄像头...")
bot = LOBOROBOT()
bot.t_stop(0)  # 确保开机静止

# 初始化云台角度
PAN_CH = 10    # 左右通道
TILT_CH = 9    # 上下通道
current_pan = 80
current_tilt = 0
bot.set_servo_angle(PAN_CH, current_pan)
bot.set_servo_angle(TILT_CH, current_tilt)

初始化底盘与摄像头...


In [3]:
# ================= 2. 视频流全局缓存机制 =================
# 全局变量，存放最新的一帧 JPEG 图像，确保多客户端/网络波动时视频不卡顿、无延迟积压
global_frame = None
#线程锁，更新画面 or 上传画面时 上锁，保证画面的完整性
frame_lock = threading.Lock()

# 视频采集子线程
def video_capture_thread():
    global global_frame
    # 优化后的 GStreamer 管道
    gstreamer_pipeline = (
        "libcamerasrc ! "
        "video/x-raw, width=640, height=480, framerate=30/1 ! "
        "videoconvert ! "
        "video/x-raw, format=BGR ! "  # 强制转换为 OpenCV 原生支持的 BGR 格式，提高效率
        "appsink drop=true max-buffers=1" # 【关键】丢弃旧帧，缓冲区只留1帧，保证画面0延迟
    )
    
    # 告诉 OpenCV 使用 GStreamer 后端来解析这个管道
    cap = cv2.VideoCapture(gstreamer_pipeline, cv2.CAP_GSTREAMER)

    # 检查是否成功打开
    if not cap.isOpened():
        print("❌ 严重错误：无法通过 GStreamer 打开摄像头！请检查管道设置。")
        return

    while True:
        ret, frame = cap.read()
        if ret:
            # 如果原始画面颠倒，翻转画面 (根据你实际摄像头的安装方向决定，-1是中心对称翻转)
            frame = cv2.flip(frame, -1)

            # 将画面编码为 JPEG 
            encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), 60]
            ret_enc, buffer = cv2.imencode('.jpg', frame, encode_param)

            if ret_enc:
                with frame_lock:
                    global_frame = buffer.tobytes()
        else:
            time.sleep(0.01)

# 启动视频采集线程
t_cam = threading.Thread(target=video_capture_thread)
t_cam.daemon = True
t_cam.start()

[0:24:30.466807312] [3477]  INFO Camera camera_manager.cpp:299 libcamera v0.0.4+22-923f5d70
[0:24:30.526433037] [3478]  INFO RPI raspberrypi.cpp:1476 Registered camera /base/soc/i2c0mux/i2c@1/ov5647@36 to Unicam device /dev/media3 and ISP device /dev/media0
[0:24:30.536846486] [3481]  INFO Camera camera.cpp:1028 configuring streams: (0) 640x480-NV21
[0:24:30.537606108] [3478]  INFO RPI raspberrypi.cpp:851 Sensor: /base/soc/i2c0mux/i2c@1/ov5647@36 - Selected sensor format: 640x480-SGBRG10_1X10 - Selected unicam format: 640x480-pGAA


In [4]:
# ================= 3. HTTP 服务器 =================
#Flask Web框架的标准开头：建立一个名为app的网站服务器程序
app = Flask(__name__)
# 关闭 Flask 默认的访问日志，避免控制台被刷屏
log = logging.getLogger('werkzeug')
log.setLevel(logging.ERROR)

def generate_stream():
    """视频流生成器，推送最新的 global_frame"""
    while True:
        with frame_lock:
            jpeg = global_frame

        if jpeg is not None:
            # 拼接 MJPEG 协议数据包
            yield (b'--frame\r\n'
                   b'Content-Type: image/jpeg\r\n\r\n' + jpeg + b'\r\n')
        # 稍微休眠，限制最高推送帧率约30fps，避免耗尽 WiFi 带宽
        time.sleep(0.03)

#若有人访问了网址，则执行video_feed函数
@app.route('/mycamera')
def video_feed():
    # 响应 HTTP 请求，类型设为 multipart/x-mixed-replace
    return Response(generate_stream(), mimetype='multipart/x-mixed-replace; boundary=frame')

In [5]:
# ================= 4. 获取本机IP并开启网络服务 (修改为 TCP) =================
def get_ip_address():
    try:
        s = socket(AF_INET, SOCK_DGRAM)
        s.connect(("8.8.8.8", 80))
        ip = s.getsockname()[0]
        s.close()
        return ip
    except error:
        return "127.0.0.1"

ip = get_ip_address()
print(f"✅ 树莓派视频流地址: http://{ip}:8080/mycamera")

# 在子线程中启动 Flask 服务器
def run_flask():
    app.run(host='0.0.0.0', port=8080, threaded=True, use_reloader=False)

t_flask = threading.Thread(target=run_flask)
t_flask.daemon = True
t_flask.start()

# 开启 TCP 指令接收服务器 (修改为 SOCK_STREAM)
tcp_server = socket(AF_INET, SOCK_STREAM)
# 设置允许端口重用，防止 Jupyter 重复运行代码时报错“端口被占用”
tcp_server.setsockopt(SOL_SOCKET, SO_REUSEADDR, 1)
tcp_server.bind(('0.0.0.0', 2001))
tcp_server.listen(1) # 开始监听连接
print(f"✅ 树莓派指令接收 TCP 端口已开启，等待连接...")

✅ 树莓派视频流地址: http://192.168.43.9:8080/mycamera
✅ 树莓派指令接收 TCP 端口已开启，等待连接...
 * Serving Flask app "__main__" (lazy loading)
 * Environment: production
   Use a production WSGI server instead.
 * Debug mode: off


In [ ]:
# ================= 5. 主循环：接收指令并执行 =================
speed = 50 # 默认速度

try:
    while True:
        # 1. 阻塞等待 PC 端软件点击“保存/连接”
        client_socket, client_addr = tcp_server.accept()
        print(f"🎉 成功连接到 PC 软件: {client_addr}")
        
        # 2. 内层循环：持续接收该客户端发来的指令
        while True:
            # 接收数据，TCP 使用 recv (1024是缓冲区大小)
            data_recv = client_socket.recv(1024)
            
            # 如果接收不到数据，说明 PC 端软件关闭了，跳出内层循环重新等待
            if not data_recv:
                print("⚠️ PC端已断开连接，等待重新连接...")
                client_socket.close()
                break 
                
            cmd = data_recv.decode('utf-8').strip()
            print(f"收到指令: {cmd}") # 建议保留打印，方便在Jupyter里观察

            # --- 运动控制 --- (保持原样)
            if cmd == "UP":          bot.t_up(speed, 0)
            elif cmd == "DOWN":      bot.t_down(speed, 0)
            elif cmd == "LEFT_MOVE": bot.moveLeft(speed, 0)
            elif cmd == "RIGHT_MOVE":bot.moveRight(speed, 0)
            elif cmd == "TURN_L":    bot.turnLeft(speed, 0)
            elif cmd == "TURN_R":    bot.turnRight(speed, 0)
            elif cmd == "UP_LEFT":   bot.forward_Left(speed, 0)
            elif cmd == "UP_RIGHT":  bot.forward_Right(speed,0)
            elif cmd == "DOWN_LEFT": bot.backward_Left(speed,0)
            elif cmd == "DOWN_RIGHT":bot.backward_Right(speed,0)
            elif cmd == "STOP":      bot.t_stop(0)

            # --- 舵机(云台)控制 --- (保持原样)
            elif cmd == "CAM_UP":
                current_tilt = max(0, current_tilt - 10)
                bot.set_servo_angle(TILT_CH, current_tilt)
            elif cmd == "CAM_DOWN":
                current_tilt = min(180, current_tilt + 10)
                bot.set_servo_angle(TILT_CH, current_tilt)
            elif cmd == "CAM_LEFT":
                current_pan = min(180, current_pan + 10)
                bot.set_servo_angle(PAN_CH, current_pan)
            elif cmd == "CAM_RIGHT":
                current_pan = max(0, current_pan - 10)
                bot.set_servo_angle(PAN_CH, current_pan)

except KeyboardInterrupt:
    print("程序被手动终止。")
finally:
    bot.t_stop(0) # 停车
    tcp_server.close() # 释放网络端口
    print("资源已安全释放。")

🎉 成功连接到 PC 软件: ('192.168.43.8', 57057)
收到指令: UP
⚠️ PC端已断开连接，等待重新连接...
🎉 成功连接到 PC 软件: ('192.168.43.8', 57058)
收到指令: UP
⚠️ PC端已断开连接，等待重新连接...
🎉 成功连接到 PC 软件: ('192.168.43.8', 57059)
收到指令: STOP
⚠️ PC端已断开连接，等待重新连接...
🎉 成功连接到 PC 软件: ('192.168.43.8', 57060)
收到指令: UP
⚠️ PC端已断开连接，等待重新连接...
🎉 成功连接到 PC 软件: ('192.168.43.8', 57061)
收到指令: STOP
⚠️ PC端已断开连接，等待重新连接...
🎉 成功连接到 PC 软件: ('192.168.43.8', 57062)
收到指令: DOWN
⚠️ PC端已断开连接，等待重新连接...
🎉 成功连接到 PC 软件: ('192.168.43.8', 57063)
收到指令: DOWN
⚠️ PC端已断开连接，等待重新连接...
🎉 成功连接到 PC 软件: ('192.168.43.8', 57064)
收到指令: STOP
⚠️ PC端已断开连接，等待重新连接...
🎉 成功连接到 PC 软件: ('192.168.43.8', 57118)
收到指令: UP
⚠️ PC端已断开连接，等待重新连接...
🎉 成功连接到 PC 软件: ('192.168.43.8', 57130)
收到指令: STOP
⚠️ PC端已断开连接，等待重新连接...
🎉 成功连接到 PC 软件: ('192.168.43.8', 57143)
收到指令: TURN_L
⚠️ PC端已断开连接，等待重新连接...
🎉 成功连接到 PC 软件: ('192.168.43.8', 57144)
收到指令: TURN_L
⚠️ PC端已断开连接，等待重新连接...
🎉 成功连接到 PC 软件: ('192.168.43.8', 57145)
收到指令: STOP
⚠️ PC端已断开连接，等待重新连接...
🎉 成功连接到 PC 软件: ('192.168.43.8', 57146)
收到指令: UP
⚠️ PC端已断开连接，等待重新连接..